# Brain Tumor Classification (MRI)

This notebook trains **standard models** on the Kaggle brain tumor MRI dataset:
- Logistic Regression
- SVM (RBF)
- Random Forest
- CNN
- ResNet50

Dataset source: `masoudnickparvar/brain-tumor-mri-dataset`


## Dataset Setup

Place dataset inside this folder:

`dataset/brain-tumor-mri-dataset/Training`

Class folders expected inside `Training`:
- glioma
- meningioma
- pituitary
- notumor (or no_tumor)


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier


In [ ]:
DATASET_DIR = Path("dataset/brain-tumor-mri-dataset/Training")
IMAGE_SIZE = 128
MAX_PER_CLASS = 1000

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "Dataset not found. Put files at brain tumor classification/dataset/brain-tumor-mri-dataset/Training"
    )

DATASET_DIR


In [ ]:
def list_images(root: Path, max_per_class=1000):
    items = []
    for class_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        count = 0
        for img in sorted(class_dir.rglob('*')):
            if img.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
                continue
            items.append((img, class_dir.name.lower()))
            count += 1
            if count >= max_per_class:
                break
    return items

items = list_images(DATASET_DIR, MAX_PER_CLASS)
print("total images:", len(items))
print("sample:", items[:3])


In [ ]:
def load_flattened(items, image_size=128):
    X, y = [], []
    for path, label in items:
        img = Image.open(path).convert('L').resize((image_size, image_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        X.append(arr.flatten())
        y.append(label)
    return np.asarray(X, dtype=np.float32), np.asarray(y)

X, y = load_flattened(items, IMAGE_SIZE)
print(X.shape, y.shape)


In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Classes:", list(le.classes_))


In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=2500, multi_class='multinomial'),
    "svm_rbf": SVC(kernel='rbf', C=4.0, gamma='scale', probability=True),
    "random_forest": RandomForestClassifier(n_estimators=240, random_state=42),
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    print("
", "="*20, name, "="*20)
    print("accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred, target_names=le.classes_, zero_division=0))
    print("confusion matrix:
", confusion_matrix(y_test, pred))


## Deep Learning Block (CNN + ResNet50)

Run this section only if TensorFlow is installed.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
    from tensorflow.keras.models import Model
    from tensorflow.keras.applications import ResNet50
except Exception as e:
    print("TensorFlow not available:", e)
    tf = None


In [ ]:
if tf is not None:
    def load_rgb(items, image_size=128):
        Xc, yc = [], []
        for path, label in items:
            img = Image.open(path).convert('RGB').resize((image_size, image_size))
            Xc.append(np.asarray(img, dtype=np.float32) / 255.0)
            yc.append(label)
        return np.asarray(Xc, dtype=np.float32), np.asarray(yc)

    Xc, yc = load_rgb(items, IMAGE_SIZE)
    y_enc_c = le.fit_transform(yc)
    Xc_train, Xc_test, yc_train, yc_test = train_test_split(
        Xc, y_enc_c, test_size=0.2, random_state=42, stratify=y_enc_c
    )

    n_classes = len(np.unique(y_enc_c))
    yc_train_oh = tf.keras.utils.to_categorical(yc_train, n_classes)
    yc_test_oh = tf.keras.utils.to_categorical(yc_test, n_classes)

    cnn = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        MaxPooling2D(2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    cnn.fit(Xc_train, yc_train_oh, epochs=6, batch_size=32, validation_split=0.15, verbose=1)
    _, cnn_acc = cnn.evaluate(Xc_test, yc_test_oh, verbose=0)
    print("CNN accuracy:", cnn_acc)

    base = ResNet50(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.3)(x)
    out = Dense(n_classes, activation='softmax')(x)
    resnet = Model(inputs=base.input, outputs=out)
    resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    resnet.fit(Xc_train, yc_train_oh, epochs=5, batch_size=24, validation_split=0.15, verbose=1)
    _, resnet_acc = resnet.evaluate(Xc_test, yc_test_oh, verbose=0)
    print("ResNet50 accuracy:", resnet_acc)
